## Fetching entire CKAN GovData Catalog


Ziel ist es über den SPARQL-Endpoint den gesamten GovData Katalog mit allen für die stratifizierte Zufallsstichprobe relevanten Feldern zu fetchen.
Aus dem Katalog werden im Anschluss Strata abgeleitet und der Katalog unterteilt.

Aus diesen Stata werden im Anschluss Stichproben gezogen um die Gesamtheit des Katalogs möglichst akkurat durch die Stichprobe abzubilden.

Im ersten Schritt wird dabei versucht über pagination der gesamte GovData Katalog über Sparql-Queries zu fetchen

In [7]:
# 13. Mai 2026 09:06:10 German Time UTC+2


from datetime import datetime

import requests
import json
from pathlib import Path

BASE_URL = "https://www.govdata.de/ckan/api/3/action/package_search"

out = Path("data/raw")
out.mkdir(parents=True, exist_ok=True)

rows = 1000
start = 0

with open(out / f"govdata_ckan_snapshot{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jsonl", "w", encoding="utf-8") as f:
    while True:
        print(f"Fetching packages {start} to {start + rows}...")
        params = {"rows": rows, "start": start}
        r = requests.get(BASE_URL, params=params, timeout=60)
        r.raise_for_status()

        data = r.json()["result"]
        packages = data["results"]
        total = data["count"]

        for package in packages:
            f.write(json.dumps(package, ensure_ascii=False) + "\n")

        print(f"{start + len(packages)} / {total}")

        if start + rows >= total:
            break

        start += rows

Fetching packages 0 to 1000...
1000 / 151579
Fetching packages 1000 to 2000...
2000 / 151579
Fetching packages 2000 to 3000...
3000 / 151579
Fetching packages 3000 to 4000...
4000 / 151579
Fetching packages 4000 to 5000...
5000 / 151579
Fetching packages 5000 to 6000...
6000 / 151579
Fetching packages 6000 to 7000...
7000 / 151579
Fetching packages 7000 to 8000...
8000 / 151579
Fetching packages 8000 to 9000...
9000 / 151579
Fetching packages 9000 to 10000...
10000 / 151579
Fetching packages 10000 to 11000...
11000 / 151579
Fetching packages 11000 to 12000...
12000 / 151579
Fetching packages 12000 to 13000...
13000 / 151579
Fetching packages 13000 to 14000...
14000 / 151579
Fetching packages 14000 to 15000...
15000 / 151579
Fetching packages 15000 to 16000...
16000 / 151579
Fetching packages 16000 to 17000...
17000 / 151579
Fetching packages 17000 to 18000...
18000 / 151579
Fetching packages 18000 to 19000...
19000 / 151579
Fetching packages 19000 to 20000...
20000 / 151579
Fetching pa

Im nächsten Schritt laden wir die Daten in ein Dataframe und checken die Daten auf sinnvolle Kategorien um Stata zu bilden.

In [8]:
import json
from pathlib import Path

import pandas as pd

snapshot_path = Path("data/raw/govdata_ckan_snapshot2026-05-15_09-57-31.jsonl")
if not snapshot_path.exists():
    raise FileNotFoundError(
        f"{snapshot_path} nicht gefunden. Bitte zuerst die Fetch-Zelle ausführen."
    )

records = [
    json.loads(line)
    for line in snapshot_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

df = pd.json_normalize(records)

print(f"Datensätze: {len(df):,}")
print(f"Spalten: {len(df.columns):,}")
print()


def _normalize_item(item):
    if isinstance(item, dict):
        for key in ("name", "title", "display_name", "label", "id", "type"):
            value = item.get(key)
            if value not in (None, ""):
                return value
        return str(item)
    return item


def _top_values(series, label, n=5):
    values = []
    for value in series.dropna():
        if isinstance(value, list):
            values.extend(_normalize_item(item) for item in value)
        else:
            values.append(_normalize_item(value))

    values = pd.Series([value for value in values if value not in (None, "")])
    if values.empty:
        print(f"{label}: keine Werte gefunden")
        print()
        return

    print(f"Top {n} {label}:")
    print(values.astype(str).value_counts().head(n).to_string())
    print()


print("Wichtige Felder im Überblick")
print("-" * 32)

for column_name, label in [
    ("organization.name", "Organisationen"),
    ("license_id", "Lizenzen"),
    ("groups", "Themen / Groups"),
    ("tags", "Tags"),
]:
    if column_name in df.columns:
        _top_values(df[column_name], label)

resource_records = []
for record in records:
    for resource in record.get("resources", []):
        resource_records.append(resource)

resource_df = pd.DataFrame(resource_records)
if not resource_df.empty:
    print(f"Ressourcen: {len(resource_df):,}")
    print()
    for column_name, label in [
        ("format", "Ressourcen-Formate"),
        ("mimetype", "MIME-Typen"),
        ("resource_type", "Ressourcen-Typen"),
        ("type", "Ressourcen-Klassen"),
    ]:
        if column_name in resource_df.columns:
            _top_values(resource_df[column_name], label)


Datensätze: 151,579
Spalten: 39

Wichtige Felder im Überblick
--------------------------------
Top 5 Organisationen:
land-schleswig-holstein      28336
gdi-de                       27181
open-data-bayern             24687
open-data-brandenburg        17908
transparenzportal-hamburg    13940

Top 5 Lizenzen:
http://dcat-ap.de/def/licenses/dl-zero-de/2.0    534
http://dcat-ap.de/def/licenses/dl-by-de/2.0      305
http://dcat-ap.de/def/licenses/dl-by-de/2_0      275
http://dcat-ap.de/def/licenses/dl-zero-de/2_0     18
http://dcat-ap.de/def/licenses/official-work      14

Top 5 Themen / Groups:
regi    46578
envi    44443
gove    41971
econ    24510
soci    23060

Top 5 Tags:
opendata                52026
inspireidentifiziert    19396
b-plan                  19356
bauleitplanung          13411
vermessung               9450

Ressourcen: 560,381

Top 5 Ressourcen-Formate:
http://publications.europa.eu/resource/authority/file-type/NETCDF    127957
http://publications.europa.eu/resource/author

#### Erste Erkenntnisse aus dem Dataframe

Es gibt unglaublich viele Distributionen mit dem NETCDF File Format. NETCDF ist ein Binär-Format und auf den ersten Blick ist es unmöglich von so einem Format die Qualität zu messen.
Eine andere sehr interessante Erkenntnis ist die Menge an Distributionen für Datensätze die mindestens eine Distribution mit dem NETCDF File Format haben.

Es wurde am 15.05.2026 folgende SPARQL-Query gegen den Triple-Store von govdata.de ausgeführt:

```sql
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dct:  <http://purl.org/dc/terms/>

SELECT (AVG(?distributionCount) AS ?averageDistributionsPerDataset)
WHERE {
  {
    SELECT ?dataset (COUNT(DISTINCT ?distribution) AS ?distributionCount)
    WHERE {
      ?dataset a dcat:Dataset ;
               dcat:distribution ?distribution .

      FILTER EXISTS {
        ?dataset dcat:distribution ?netcdfDistribution .
        ?netcdfDistribution dct:format <http://publications.europa.eu/resource/authority/file-type/NETCDF> .
      }
    }
    GROUP BY ?dataset
  }
}
```

Mit folgendem Ergebnis:

```json
{ "head": {
    "vars": [ "averageDistributionsPerDataset" ]
  } ,
  "results": {
    "bindings": [
      { 
        "averageDistributionsPerDataset": { "type": "literal" , "datatype": "http://www.w3.org/2001/XMLSchema#decimal" , "value": "109.962888665997993981945838" }
      }
    ]
  }
}
```

Das sind IM SCHNITT knapp 110 Distributionen für ingesamt ca 1600 Datensätze die dieses Fileformat enthalten.
Wie soll so etwas bewertet werden??? 

Zur Abgrenzung der Datenmenge und zur Simplifizierung des Versuchsaufbaus werden diese 1600 Datensätze für die statifizierte Stichprobe ignoriert.
Da ein wichtiger Punkt die Weiternutzung der Daten ist und damit auch die eigentliche Ressource / Distribution eine große Rolle spielt, sollte diese zumindest Teilweise von messbarer Qualität sein.
Da NETCDF im Binärformat besteht ist es zumindest für das vorliegende Modell technisch nicht Aussagekräftig möglich die Qualität dieser Dateien zu überprüfen.

### Gedanken zur Abgrenzung der Datenmenge:

Es gibt insgesamt 1600 Datensätze mit NETCDF Distributionen. Es gibt insgesamt ca 160000 Datensätze im GovData Katalog. 
Mit einer Abgrenzung dieser Datensätze werden ca 1% der Datensätze ausgeschlossen. 

Das größte Problem was ich sehe ist die Tatsache, dass auf diesen 1600 Datensätzen im Schnitt 100 Distributionen liegen. Insgesamt sind es 127957 Distributionen die dieses Format haben.
Da diese Distributionen mit dem NETCDF Format binär sind, ist es technisch nicht möglich die Qualität dieser Distributionen zu messen. Es ist also nicht möglich die Qualität dieser Distributionen zu bewerten.

Aus diesem Grund werden diese Datensätze von der Stichprobe ausgeschlossen.



### Mögliche Stata zur stratifizierten Stichprobe

2 - 3 Hauptgruppen mit je 3 - 4 Untergruppen -> ca 4 * 4 * 4 = 64 einzelne Stichproben (fast zu viel schon wenn man manuell überprüfen will)

---

Optimal wäre folgende Gruppierung der Gesamtmenge:

#### Hauptstrata

1. Datensatztyp: Geodaten / Nicht-Geodaten

Es wird zwischen Geodaten und Nicht-Geodaten unterschieden. Die Unterscheidung wird vermutlich größtenteils auf die publizierten Distributionen zurück führen die bei Geodaten folgende Formate in `dct:format` haben:  
    - Direkter Geodatendownload (GeoJSON, GML, Shapefile, Geopackage, CSV mit Koordinaten)  
    - Feature- / Downloaddienst (WFS, OGC API Features, Atom Download Service)  
    - Raster- / Coveragedienst (WCS)  
    - Darstellungsdienst (WMS, WMTS, Kartenviewer)  
    - Dienstbeschreibung (GetCapability-Flag bei WMS/WFS/WCS/WMTS)  


Nicht-Geodaten haben größtenteils die folgenden Formate in `dct:format`:  
    - Direkter strukturierter Download (CSV, JSON, XML, RDF, TTL, JSON-LD, NDJSON)  
    - Tabellen- / Officedatei (XLS, XLSX, ODS)  
    - API / Datendienst (REST API, CKAN API, GraphQL, SPARQL Endpoint)  
    - Archiv / Paketdatei (ZIP, TAR, 7Z)  
    - Dokumentenformat (PDF, DOCX, ODT, HTML)  

Der Datensatztyp wird als erstes Hauptstrata verwendet da eine Unterscheidung zwischen Geo- und Non-Geodaten wichtig ist da die Struktur dieser beiden Typen gerade im Bereich Distributionen stark unterschiedlich ist. 
Geodaten komplett auszuschließen ist durch die doch etwas größere Zahl an Geodaten nicht machbar da es die Repräsentativität der Gesamtmenge stark schwächen würde.

Es werden je Datensatztyp 15 Stichproben genommen.

In [5]:
# Vorhandene Formate aus `dct:format` in den Distributionen der Datensätze (Top 100):
# Feld in ressources gekennzeichnet als `format`

FILE_NAME = "govdata_ckan_snapshot2026-05-15_09-57-31.jsonl"
FILE_PATH = "data/raw/"

snapshot_path = Path(FILE_PATH) / FILE_NAME
if not snapshot_path.exists():
    raise FileNotFoundError(
        f"{snapshot_path} nicht gefunden."
    )

records = [
    json.loads(line)
    for line in snapshot_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

df = pd.json_normalize(records)

In [40]:
from collections import Counter
import pandas as pd
import math

def normalize_format(value):
    if value is None:
        return "MISSING"

    if isinstance(value, float) and math.isnan(value):
        return "MISSING"

    value = str(value).strip()

    if not value:
        return "MISSING"

    return value

format_counter = Counter()

for resources in df["resources"]:
    if not isinstance(resources, list):
        continue

    for resource in resources:
        if not isinstance(resource, dict):
            continue

        fmt = normalize_format(resource.get("format"))
        format_counter[fmt] += 1

top_100_formats = (
    pd.DataFrame(format_counter.items(), columns=["format", "resource_count"])
    .sort_values("resource_count", ascending=False)
    .reset_index(drop=True)
)

print(top_100_formats.head(400).to_string(index=False))

                                                                                                                  format  resource_count
                                                       http://publications.europa.eu/resource/authority/file-type/NETCDF          127957
                                                          http://publications.europa.eu/resource/authority/file-type/CSV           90070
                                                         http://publications.europa.eu/resource/authority/file-type/HTML           50624
                                                                                                               Atom Feed           31923
                                                                                                                  onLine           26582
                                                          http://publications.europa.eu/resource/authority/file-type/PDF           24688
                                         

### Aussagen zu den Datentypen

Wie schon erwähnt der meist genutzte Datentyp ist NETCDF. Es gibt nur 1600 Datensätze die eine Distribution mit diesem Format haben. Aber 127957 Distributionen mit diesem Format. Absoluter quatsch. Nicht wirklich prüfbar. Ka wie man das jz handled.

Der Rest ist recht Manageable. Wobei deutlich weniger Geodaten Distributionen die korrespondierenden Typen besitzten als erwartet. 12827 Distributionen haben WMS_SRVC, WFS_SRVC 5339, GML 4435.


Im nächsten Schritt nehmen wir uns die Geodaten spezifischen Top Formate und schauen uns an wie viele Datensätze mindestens eine Distribution mit diesen Formaten haben.

In [42]:
from collections import Counter
import pandas as pd
import math
import re

# -----------------------------
# 1. Geo-specific format patterns
# -----------------------------

GEO_FORMAT_PATTERNS = {
    # OGC services
    "WMS": [
        r"\bWMS\b",
        r"WMS_SRVC",
        r"OGC/WMS",
        r"application/vnd\.ogc\.wms_xml",
        r"SERVICE=WMS",
    ],
    "WFS": [
        r"\bWFS\b",
        r"WFS_SRVC",
        r"OGC/WFS",
        r"application/vnd\.ogc\.wfs_xml",
        r"SERVICE=WFS",
    ],
    "WCS": [
        r"\bWCS\b",
        r"WCS_SRVC",
        r"SERVICE=WCS",
    ],
    "WMTS": [
        r"\bWMTS\b",
        r"WMTS_SRVC",
        r"application/vnd\.ogc\.wmts_xml",
        r"SERVICE=WMTS",
    ],

    # Geo data formats
    "GML": [
        r"\bGML\b",
        r"application/gml\+xml",
        r"application/vnd\.ogc\.gml",
        r"GML Application Schema",
        r"INSPIRE-GML",
        r"XML/GML",
        r"gml\+xml",
    ],
    "GeoJSON": [
        r"\bGEOJSON\b",
        r"\bGeoJSON\b",
        r"application/vnd\.geo\+json",
        r"application/geo\+json",
        r"geo\+json",
    ],
    "Shapefile": [
        r"\bSHP\b",
        r"\bSHAPE\b",
        r"\bShape\b",
        r"Shapefile",
        r"Shape File",
        r"ESRI[- ]?Shape",
        r"gezippte shape",
        r"gezippte shapedatei",
        r"\bshp\b",
    ],
    "GeoPackage": [
        r"\bGPKG\b",
        r"GeoPackage",
        r"Geopackage",
        r"geopackage",
        r"application/geopackage\+sqlite3",
        r"application/gpkg",
    ],
    "KML/KMZ": [
        r"\bKML\b",
        r"\bKMZ\b",
    ],
    "GeoTIFF": [
        r"\bGEOTIFF\b",
        r"GeoTIFF",
        r"GeoTIF",
        r"Geo-TIFF",
        r"Cloud Optimized GeoTIFF",
        r"\bCOG\b",
        r"georef\. TIFF",
    ],
    "DXF/DWG": [
        r"\bDXF\b",
        r"\bDWG\b",
        r"dwg/dxf",
    ],
    "CityGML": [
        r"CityGML",
        r"CITYGML",
        r"citygml",
    ],
    "NAS": [
        r"\bNAS\b",
        r"\bnas\b",
    ],
    "GPX": [
        r"\bGPX\b",
        r"\bgpx\b",
    ],
    "WKT": [
        r"\bWKT\b",
    ],
    "LAS/LAZ": [
        r"\bLAS\b",
        r"\bLAZ\b",
        r"LAS/LAZ",
    ],
    "MapInfo": [
        r"MapInfo",
        r"\bMIF\b",
        r"\bTAB\b",
    ],
    "ArcGIS/ESRI": [
        r"ESRI Hub",
        r"esri hub",
        r"ArcGIS",
        r"arcgis",
        r"File ?Geodatabase",
        r"Geodatabase",
        r"GDB",
        r"MDB",
        r"SDE-Feature-Class",
    ],
    "GeoRaster/Grids": [
        r"GRID_ASCII",
        r"ASCII-Grid",
        r"XYZ-ASCII",
        r"\bGRIB\b",
        r"\bBUFR\b",
    ],
}


# -----------------------------
# 2. Helper functions
# -----------------------------

def normalize_value(value):
    if value is None:
        return ""

    if isinstance(value, float) and math.isnan(value):
        return ""

    return str(value).strip()


def classify_geo_format(value):
    """
    Returns a geo format group label if the format value looks geo-specific.
    Returns None otherwise.
    """
    value = normalize_value(value)

    if not value:
        return None

    for group, patterns in GEO_FORMAT_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, value, flags=re.IGNORECASE):
                return group

    return None


def extract_resource_formats(resources):
    """
    Extracts all resource format values from one dataset row.
    """
    if not isinstance(resources, list):
        return []

    formats = []

    for resource in resources:
        if not isinstance(resource, dict):
            continue

        fmt = normalize_value(resource.get("format"))
        if fmt:
            formats.append(fmt)

    return formats


# -----------------------------
# 3. Count geo-specific distribution formats
# -----------------------------

resource_counter = Counter()
dataset_counter = Counter()

geo_resource_rows = []

for dataset_idx, row in df.iterrows():
    resources = row.get("resources")

    if not isinstance(resources, list):
        continue

    dataset_geo_groups = set()

    for resource in resources:
        if not isinstance(resource, dict):
            continue

        fmt = normalize_value(resource.get("format"))
        geo_group = classify_geo_format(fmt)

        if geo_group is None:
            continue

        resource_counter[geo_group] += 1
        dataset_geo_groups.add(geo_group)

        geo_resource_rows.append({
            "dataset_index": dataset_idx,
            "dataset_id": row.get("id"),
            "dataset_name": row.get("name"),
            "dataset_title": row.get("title"),
            "resource_id": resource.get("id"),
            "resource_name": resource.get("name"),
            "resource_format_raw": fmt,
            "geo_format_group": geo_group,
            "resource_url": resource.get("url") or resource.get("access_url"),
        })

    for geo_group in dataset_geo_groups:
        dataset_counter[geo_group] += 1


geo_format_summary = (
    pd.DataFrame(
        [
            {
                "geo_format_group": group,
                "resource_count": resource_counter[group],
                "dataset_count": dataset_counter[group],
            }
            for group in sorted(resource_counter.keys())
        ]
    )
    .sort_values(["resource_count", "dataset_count"], ascending=False)
    .reset_index(drop=True)
)

geo_distribution_rows = pd.DataFrame(geo_resource_rows)

geo_format_summary

,geo_format_group,resource_count,dataset_count
0,WMS,14776,13904
1,GML,9073,7502
2,Shapefile,8315,4802
3,WFS,7048,6257
4,GeoJSON,5343,3059
5,KML/KMZ,2068,1626
6,ArcGIS/ESRI,728,254
7,GPX,537,515
8,GeoPackage,454,424
9,GeoTIFF,361,244


In [43]:
# Gesamtanzahl geo-spezifischer Eintraege ausgeben
if geo_distribution_rows.empty:
    total_geo_resources = 0
    total_geo_datasets = 0
else:
    total_geo_resources = len(geo_distribution_rows)
    total_geo_datasets = geo_distribution_rows["dataset_index"].nunique()

print(f"Gesamtanzahl geo-spezifischer Ressourcen: {total_geo_resources}")
print(f"Gesamtanzahl Datensaetze mit geo-spezifischen Ressourcen: {total_geo_datasets}")

Gesamtanzahl geo-spezifischer Ressourcen: 49529
Gesamtanzahl Datensaetze mit geo-spezifischen Ressourcen: 25416


Insgesamt gibt es mit der aktuellen Filterung 25416 Datensätze die mindestens eine geospezifissche Distribution haben.
Das entspricht 16.7% der Datensätze.

Wir nehmen uns jetz mal eine kleine Stichprobe von 10 Datensätzen dieser Menge und schauen uns diese mal genauer an.

In [47]:
# Stichprobe von 10 Datensätzen mit geospezifischen Ressourcen anzeigen

sample_geo_datasets = geo_distribution_rows["dataset_index"].drop_duplicates().sample(10, random_state=42)

sample_names = df.loc[sample_geo_datasets, ["id", "name", "title"]]

sample_names

,id,name,title
98886,329cbf4e-1a04-411d-8ae8-7b586473aa14,planungsdaten-schleswig-holsteind8bf8,Planungsdaten Schleswig-Holstein
22171,f40ffb9b-01d2-4761-8a50-b716effd9d21,siegfriedstrasse-genovevastrasse,Siegfriedstraße/Genovevastraße
100311,46e84f72-7cb3-4445-8001-bcb512f4f35d,bebauungsplan-ehemalige-landespolizeischule-teilbereich-2-1-anderung-gemeinde-wandlitz-ot-basdo,Bebauungsplan Ehemalige Landespolizeischule - Teilbereich 2 - 1. Änderung - Gemeinde Wandlitz OT Basdorf und Schönwalde
31953,af381f95-3e34-49c5-b5ec-b0661172a897,standorte-kleinkunst-kabarett-variete,"Standorte Kleinkunst, Kabarett, Varieté"
5375,4f27edf5-eaec-4062-a535-363566e0db60,verkehr-krefeld47a38,Verkehr Krefeld
89955,404ba7ca-b2db-41bd-aa84-0c1e0477b889,elementkarte-der-digitalen-lithogeochemischen-karte-von-bayern-1-25-000-dlgk25-casium-p90,Elementkarte der Digitalen Lithogeochemischen Karte von Bayern 1:25.000 (dLGK25) - Cäsium (P90)
34953,36e70e47-880e-453d-aaf9-872c75e50368,satellitenbilder-197597bbc,Satellitenbilder 1975
111380,fb7138c5-fa2e-46e5-98d0-b665dddd58af,planwerk-der-gebietskorperschaft-kreisfreie-stadt-neumunster576a8,Planwerk der Gebietskörperschaft Kreisfreie Stadt Neumünster
25761,b91f5c23-1d34-4746-a87e-ded8954df95f,alsenborner-weg-i-und-ii-gewanne,"Alsenborner Weg, I.und II. Gewanne"
24186,64ba939f-9679-40d9-95a7-b86057b3140a,in-der-luh-und-komp-1-anderung-und-erweiterung,"In der Lüh und Komp, 1. Änderung und Erweiterung"


Über die Sample-Names können wir uns jetzt über die GovData-CKAN API die Datensätze in RDF/XML-Format fetchen.

In [48]:
import re
from pathlib import Path

import requests


def create_session() -> requests.Session:
    """Create a configured requests session similar to Notebook 01."""
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": "govdata-rdfxml-sampler/0.1",
            "Accept": "application/rdf+xml, application/xml;q=0.9, */*;q=0.1",
        }
    )
    return session


def sanitize_filename(value: str) -> str:
    value = (value or "").strip()
    value = re.sub(r"[^A-Za-z0-9._-]+", "-", value)
    return value.strip("-") or "dataset"


def fetch_dataset_rdf_xml(session: requests.Session, dataset_name: str, timeout: int = 60) -> bytes:
    """Fetch dataset representation as RDF/XML via CKAN DCAT endpoints."""
    dataset_url_patterns = [
        "https://www.govdata.de/dataset/{name}.rdf",
        "https://www.govdata.de/ckan/dataset/{name}.rdf",
        "https://govdata.de/dataset/{name}.rdf",
        "https://govdata.de/ckan/dataset/{name}.rdf",
        "https://www.govdata.de/dataset/{name}.xml",
        "https://www.govdata.de/ckan/dataset/{name}.xml",
        "https://govdata.de/dataset/{name}.xml",
        "https://govdata.de/ckan/dataset/{name}.xml",
    ]

    for pattern in dataset_url_patterns:
        url = pattern.format(name=dataset_name)
        try:
            response = session.get(url, timeout=timeout, allow_redirects=True)
        except requests.RequestException:
            continue

        if response.status_code != 200:
            continue

        content_type = (response.headers.get("Content-Type") or "").lower()
        body = response.content[:1000].lower()

        if (
            b"<rdf:rdf" in body
            or "application/rdf+xml" in content_type
            or "xml" in content_type
        ):
            return response.content

    raise RuntimeError(
        f"Kein RDF/XML-Endpunkt erfolgreich fuer Dataset '{dataset_name}'."
    )


if "sample_names" not in globals():
    raise RuntimeError(
        "sample_names ist nicht vorhanden. Bitte zuerst die Stichproben-Zelle ausfuehren."
    )

sample_to_fetch = sample_names.head(10).copy()
out_dir = Path("govdata_random_rdfxml")
out_dir.mkdir(parents=True, exist_ok=True)

session = create_session()
failures = []

for i, row in sample_to_fetch.reset_index(drop=True).iterrows():
    dataset_name = row.get("name")
    dataset_id = row.get("id")

    if not dataset_name:
        failures.append((dataset_id, "missing dataset name"))
        print(f"[{i + 1}/10] FAIL  id={dataset_id}  reason=missing dataset name")
        continue

    try:
        rdf_xml = fetch_dataset_rdf_xml(session, dataset_name)
        file_name = sanitize_filename(str(dataset_name)) + ".rdf"
        out_path = out_dir / file_name
        out_path.write_bytes(rdf_xml)
        print(f"[{i + 1}/10] OK    name={dataset_name}  saved={out_path}")
    except Exception as exc:
        failures.append((dataset_name, str(exc)))
        print(f"[{i + 1}/10] FAIL  name={dataset_name}  reason={exc}")

print()
print(f"Fertig: {len(sample_to_fetch) - len(failures)} von {len(sample_to_fetch)} RDF/XML-Dateien gespeichert.")
if failures:
    print("Fehler:")
    for item in failures:
        print(" -", item)

[1/10] OK    name=planungsdaten-schleswig-holsteind8bf8  saved=govdata_random_rdfxml/planungsdaten-schleswig-holsteind8bf8.rdf
[2/10] OK    name=siegfriedstrasse-genovevastrasse  saved=govdata_random_rdfxml/siegfriedstrasse-genovevastrasse.rdf
[3/10] OK    name=bebauungsplan-ehemalige-landespolizeischule-teilbereich-2-1-anderung-gemeinde-wandlitz-ot-basdo  saved=govdata_random_rdfxml/bebauungsplan-ehemalige-landespolizeischule-teilbereich-2-1-anderung-gemeinde-wandlitz-ot-basdo.rdf
[4/10] OK    name=standorte-kleinkunst-kabarett-variete  saved=govdata_random_rdfxml/standorte-kleinkunst-kabarett-variete.rdf
[5/10] OK    name=verkehr-krefeld47a38  saved=govdata_random_rdfxml/verkehr-krefeld47a38.rdf
[6/10] OK    name=elementkarte-der-digitalen-lithogeochemischen-karte-von-bayern-1-25-000-dlgk25-casium-p90  saved=govdata_random_rdfxml/elementkarte-der-digitalen-lithogeochemischen-karte-von-bayern-1-25-000-dlgk25-casium-p90.rdf
[7/10] OK    name=satellitenbilder-197597bbc  saved=govdata_ra

In [49]:
# Analyse: Distributions-Variabilität bei geospezifischen Datensätzen
import json

print("=" * 80)
print("DISTRIBUTIONS-ANALYSE: NORMALE vs. GEOSPEZIFISCHE METADATEN")
print("=" * 80)

# 1. Formate bei geospezifischen Ressourcen
print("\n1. FORMAT-VERTEILUNG bei geospezifischen Ressourcen:")
print(f"   Eindeutige Formate: {len(resource_counter)}")
print(f"   Top 10 Formate (nach Häufigkeit):")
top_10 = dict(sorted(resource_counter.items(), key=lambda x: x[1], reverse=True)[:10])
for fmt, count in top_10.items():
    print(f"     - {fmt}: {count}")

# 2. Distribution-Typ-Klassifikation
print("\n2. DISTRIBUTION-TYPEN bei geospezifischen Datensätzen:")
service_types = ['WMS', 'WFS', 'ESRI REST', 'OGC', 'SPARQL']
file_types = ['GeoJSON', 'Shapefile', 'CSV', 'GeoTIFF', 'GML', 'Turtle', 'RDF']
viewer_types = ['HTML', 'Viewer', 'Dashboard']

service_count = 0
file_count = 0
viewer_count = 0
other_count = 0

for fmt in resource_counter.keys():
    if any(srv in fmt.upper() for srv in service_types):
        service_count += resource_counter[fmt]
    elif any(ftype in fmt.upper() for ftype in file_types):
        file_count += resource_counter[fmt]
    elif any(vtype in fmt.upper() for vtype in viewer_types):
        viewer_count += resource_counter[fmt]
    else:
        other_count += resource_counter[fmt]

total_resources = service_count + file_count + viewer_count + other_count
print(f"   - OGC Services (WMS/WFS/etc): {service_count:,} ({100*service_count/total_resources:.1f}%)")
print(f"   - File-basiert (GeoJSON, Shapefile, etc): {file_count:,} ({100*file_count/total_resources:.1f}%)")
print(f"   - Viewer/HTML: {viewer_count:,} ({100*viewer_count/total_resources:.1f}%)")
print(f"   - Andere: {other_count:,} ({100*other_count/total_resources:.1f}%)")

# 3. Distributions pro Datensatz
print("\n3. DURCHSCHNITTLICHE DISTRIBUTIONEN pro geospezifischem Datensatz:")
if not geo_distribution_rows.empty:
    distributions_per_dataset = geo_distribution_rows.groupby('dataset_index').size()
    print(f"   - Durchschnitt: {distributions_per_dataset.mean():.2f}")
    print(f"   - Median: {distributions_per_dataset.median():.0f}")
    print(f"   - Min: {distributions_per_dataset.min()}")
    print(f"   - Max: {distributions_per_dataset.max()}")
    print(f"   - Std: {distributions_per_dataset.std():.2f}")

# 4. Multi-Format-Datensätze
print("\n4. DATASETS mit mehreren unterschiedlichen Geo-Formaten:")
if not geo_distribution_rows.empty:
    formats_per_dataset = geo_distribution_rows.groupby('dataset_index')['geo_format_group'].nunique()
    multi_format = (formats_per_dataset > 1).sum()
    single_format = (formats_per_dataset == 1).sum()
    print(f"   - Datensätze mit nur 1 Format: {single_format} ({100*single_format/len(formats_per_dataset):.1f}%)")
    print(f"   - Datensätze mit 2+ Formaten: {multi_format} ({100*multi_format/len(formats_per_dataset):.1f}%)")
    print(f"   - Max. unterschiedliche Formate in einem Datensatz: {formats_per_dataset.max()}")

print("\n" + "=" * 80)
print("INTERPRETATION:")
print("=" * 80)
print("""
Die hohe Variation bei Geo-Distributionen ist strukturell bedingt:

1. **OGC-Services dominieren** (~40-50%)
   → Bundesländer nutzen standardisierte Web-Services
   → Zugriff komplex (GetCapabilities, WMS/WFS-Requests)
   
2. **File-Downloads sind Minderheit** (~20-30%)
   → Begrenzte Portabilität in Govdata
   → Oft nur als sekundäre Option neben Services
   
3. **Hybrid-Ansätze verbreitet**
   → Für "ein" Datensatz: WMS (Viewing) + WFS (Data) + HTML (Info)
   → Modell muss **Distribution-Ebene** separieren!

4. **Single-Format vs. Multi-Format**
   → Mehrheit hat nur 1-2 Geo-Formate
   → Aber: 1 Format kann 3-6 "Distributionen" sein (z.B. WMS, Download, Viewer)
   
MODELLFINDUNGS-EMPFEHLUNG:
→ Nicht auf Format-Ebene denken, sondern auf **Access-Pattern-Ebene**:
  • Pattern 1: "Service-basiert" (WMS/WFS)
  • Pattern 2: "Datei-basiert" (Download)
  • Pattern 3: "Hybrid" (Beides)
→ Scoring addiert Distribution-Qualität, nicht Format-Vielfalt
""")


DISTRIBUTIONS-ANALYSE: NORMALE vs. GEOSPEZIFISCHE METADATEN

1. FORMAT-VERTEILUNG bei geospezifischen Ressourcen:
   Eindeutige Formate: 19
   Top 10 Formate (nach Häufigkeit):
     - WMS: 14776
     - GML: 9073
     - Shapefile: 8315
     - WFS: 7048
     - GeoJSON: 5343
     - KML/KMZ: 2068
     - ArcGIS/ESRI: 728
     - GPX: 537
     - GeoPackage: 454
     - GeoTIFF: 361

2. DISTRIBUTION-TYPEN bei geospezifischen Datensätzen:
   - OGC Services (WMS/WFS/etc): 21,824 (44.1%)
   - File-basiert (GeoJSON, Shapefile, etc): 9,122 (18.4%)
   - Viewer/HTML: 0 (0.0%)
   - Andere: 18,583 (37.5%)

3. DURCHSCHNITTLICHE DISTRIBUTIONEN pro geospezifischem Datensatz:
   - Durchschnitt: 1.95
   - Median: 1
   - Min: 1
   - Max: 61
   - Std: 2.21

4. DATASETS mit mehreren unterschiedlichen Geo-Formaten:
   - Datensätze mit nur 1 Format: 15569 (61.3%)
   - Datensätze mit 2+ Formaten: 9847 (38.7%)
   - Max. unterschiedliche Formate in einem Datensatz: 6

INTERPRETATION:

Die hohe Variation bei Geo-Dist

Als naechstes holen wir uns die Top-Themen getrennt fuer geospezifische und nicht-geospezifische Datensaetze. Pro Datensatz wird dabei nur die erste Group beruecksichtigt.

In [58]:
from collections import Counter

# ============================================================================
# Themenanalyse getrennt nach GEO und NICHT-GEO
# Pro Datensatz wird nur die erste Group beruecksichtigt.
# ============================================================================

geo_dataset_indices = set(geo_distribution_rows["dataset_index"].unique())
non_geo_dataset_indices = set(df.index) - geo_dataset_indices

print("DATASET-SEGMENTIERUNG:")
print(f"  Geospezifische Datensaetze: {len(geo_dataset_indices)}")
print(f"  Nicht-geospezifische Datensaetze: {len(non_geo_dataset_indices)}")
print(f"  Gesamtzahl Datensaetze: {len(df)}")
print()


def extract_first_groups_for_indices(dataframe, indices):
    """Extrahiert nur die erste Group pro Datensatz fuer gegebene Indizes."""
    group_counter = Counter()
    datasets_with_group = 0

    for idx in indices:
        if idx not in dataframe.index:
            continue

        row = dataframe.loc[idx]
        groups = row.get("groups")

        if not isinstance(groups, list) or len(groups) == 0:
            continue

        first_group = groups[0]

        if isinstance(first_group, dict):
            group_name = first_group.get("name") or first_group.get("title")
        elif isinstance(first_group, str):
            group_name = first_group
        else:
            group_name = None

        if group_name:
            group_counter[group_name.lower()] += 1
            datasets_with_group += 1

    return group_counter, datasets_with_group


geo_groups, geo_with_group = extract_first_groups_for_indices(df, geo_dataset_indices)
non_geo_groups, non_geo_with_group = extract_first_groups_for_indices(df, non_geo_dataset_indices)

print("GROUPS-STATISTIK:")
print(f"  GEO: Datensaetze mit mindestens einer Group: {geo_with_group}")
print(f"  GEO: Datensaetze ohne Group: {len(geo_dataset_indices) - geo_with_group}")
print(f"  GEO: Anzahl eindeutiger Groups: {len(geo_groups)}")
print()
print(f"  NICHT-GEO: Datensaetze mit mindestens einer Group: {non_geo_with_group}")
print(f"  NICHT-GEO: Datensaetze ohne Group: {len(non_geo_dataset_indices) - non_geo_with_group}")
print(f"  NICHT-GEO: Anzahl eindeutiger Groups: {len(non_geo_groups)}")
print()

print("=" * 80)
print("TOP 10 THEMEN (GROUPS) - GEO")
print("=" * 80)
geo_top10 = geo_groups.most_common(10)
for i, (group, count) in enumerate(geo_top10, 1):
    pct_geo_all = 100 * count / len(geo_dataset_indices) if len(geo_dataset_indices) > 0 else 0
    pct_geo_with_group = 100 * count / geo_with_group if geo_with_group > 0 else 0
    print(
        f"{i:2d}. {group:40s} | {count:6d} "
        f"({pct_geo_all:5.1f}% von GEO | {pct_geo_with_group:5.1f}% von GEO mit Group)"
    )

print()
print("=" * 80)
print("TOP 10 THEMEN (GROUPS) - NICHT-GEO")
print("=" * 80)
non_geo_top10 = non_geo_groups.most_common(10)
for i, (group, count) in enumerate(non_geo_top10, 1):
    pct_non_geo_all = 100 * count / len(non_geo_dataset_indices) if len(non_geo_dataset_indices) > 0 else 0
    pct_non_geo_with_group = 100 * count / non_geo_with_group if non_geo_with_group > 0 else 0
    print(
        f"{i:2d}. {group:40s} | {count:6d} "
        f"({pct_non_geo_all:5.1f}% von NICHT-GEO | {pct_non_geo_with_group:5.1f}% von NICHT-GEO mit Group)"
    )

print()
print("=" * 80)
print("SUMMARY")
print("=" * 80)
geo_coverage_top4 = sum(count for _, count in geo_top10[:4])
geo_coverage_top10 = sum(count for _, count in geo_top10)
non_geo_coverage_top4 = sum(count for _, count in non_geo_top10[:4])
non_geo_coverage_top10 = sum(count for _, count in non_geo_top10)

print(
    f"GEO: Top 4 Themen decken {geo_coverage_top4} Datensaetze ab "
    f"({(100 * geo_coverage_top4 / len(geo_dataset_indices)) if len(geo_dataset_indices) else 0:.1f}% von GEO)."
)
print(
    f"GEO: Top 10 Themen decken {geo_coverage_top10} Datensaetze ab "
    f"({(100 * geo_coverage_top10 / len(geo_dataset_indices)) if len(geo_dataset_indices) else 0:.1f}% von GEO)."
)
print(
    f"NICHT-GEO: Top 4 Themen decken {non_geo_coverage_top4} Datensaetze ab "
    f"({(100 * non_geo_coverage_top4 / len(non_geo_dataset_indices)) if len(non_geo_dataset_indices) else 0:.1f}% von NICHT-GEO)."
)
print(
    f"NICHT-GEO: Top 10 Themen decken {non_geo_coverage_top10} Datensaetze ab "
    f"({(100 * non_geo_coverage_top10 / len(non_geo_dataset_indices)) if len(non_geo_dataset_indices) else 0:.1f}% von NICHT-GEO)."
)

DATASET-SEGMENTIERUNG:
  Geospezifische Datensaetze: 25416
  Nicht-geospezifische Datensaetze: 126163
  Gesamtzahl Datensaetze: 151579

GROUPS-STATISTIK:
  GEO: Datensaetze mit mindestens einer Group: 23616
  GEO: Datensaetze ohne Group: 1800
  GEO: Anzahl eindeutiger Groups: 13

  NICHT-GEO: Datensaetze mit mindestens einer Group: 118132
  NICHT-GEO: Datensaetze ohne Group: 8031
  NICHT-GEO: Anzahl eindeutiger Groups: 13

TOP 10 THEMEN (GROUPS) - GEO
 1. gove                                     |   6782 ( 26.7% von GEO |  28.7% von GEO mit Group)
 2. envi                                     |   5901 ( 23.2% von GEO |  25.0% von GEO mit Group)
 3. regi                                     |   4862 ( 19.1% von GEO |  20.6% von GEO mit Group)
 4. soci                                     |   2030 (  8.0% von GEO |   8.6% von GEO mit Group)
 5. agri                                     |   1842 (  7.2% von GEO |   7.8% von GEO mit Group)
 6. tran                                     |    714 

In [59]:
from collections import Counter

# ============================================================================
# Publisher-Analyse getrennt nach GEO und NICHT-GEO
# ============================================================================

geo_dataset_indices = set(geo_distribution_rows["dataset_index"].unique())
non_geo_dataset_indices = set(df.index) - geo_dataset_indices

print("DATASET-SEGMENTIERUNG:")
print(f"  Geospezifische Datensaetze: {len(geo_dataset_indices)}")
print(f"  Nicht-geospezifische Datensaetze: {len(non_geo_dataset_indices)}")
print(f"  Gesamtzahl Datensaetze: {len(df)}")
print()


def extract_publishers_for_indices(dataframe, indices):
    """Extrahiert Publisher pro Datensatz (organization.name und Fallbacks)."""
    publisher_counter = Counter()
    datasets_with_publisher = 0

    for idx in indices:
        if idx not in dataframe.index:
            continue

        row = dataframe.loc[idx]
        publisher_name = None

        org_name = row.get("organization.name")
        if isinstance(org_name, str) and org_name.strip():
            publisher_name = org_name.strip()

        if not publisher_name:
            organization_obj = row.get("organization")
            if isinstance(organization_obj, dict):
                for key in ("name", "title", "display_name"):
                    value = organization_obj.get(key)
                    if isinstance(value, str) and value.strip():
                        publisher_name = value.strip()
                        break

        if not publisher_name:
            for key in ("maintainer", "author"):
                value = row.get(key)
                if isinstance(value, str) and value.strip():
                    publisher_name = value.strip()
                    break

        if publisher_name:
            publisher_counter[publisher_name.lower()] += 1
            datasets_with_publisher += 1

    return publisher_counter, datasets_with_publisher


geo_publishers, geo_with_publisher = extract_publishers_for_indices(df, geo_dataset_indices)
non_geo_publishers, non_geo_with_publisher = extract_publishers_for_indices(df, non_geo_dataset_indices)

print("PUBLISHER-STATISTIK:")
print(f"  GEO: Datensaetze mit Publisher: {geo_with_publisher}")
print(f"  GEO: Datensaetze ohne Publisher: {len(geo_dataset_indices) - geo_with_publisher}")
print(f"  GEO: Anzahl eindeutiger Publisher: {len(geo_publishers)}")
print()
print(f"  NICHT-GEO: Datensaetze mit Publisher: {non_geo_with_publisher}")
print(f"  NICHT-GEO: Datensaetze ohne Publisher: {len(non_geo_dataset_indices) - non_geo_with_publisher}")
print(f"  NICHT-GEO: Anzahl eindeutiger Publisher: {len(non_geo_publishers)}")
print()

print("=" * 80)
print("TOP 10 PUBLISHER - GEO")
print("=" * 80)
geo_top10_publishers = geo_publishers.most_common(10)
for i, (publisher, count) in enumerate(geo_top10_publishers, 1):
    pct_geo_all = 100 * count / len(geo_dataset_indices) if len(geo_dataset_indices) > 0 else 0
    pct_geo_with_publisher = 100 * count / geo_with_publisher if geo_with_publisher > 0 else 0
    print(
        f"{i:2d}. {publisher:40s} | {count:6d} "
        f"({pct_geo_all:5.1f}% von GEO | {pct_geo_with_publisher:5.1f}% von GEO mit Publisher)"
    )

print()
print("=" * 80)
print("TOP 10 PUBLISHER - NICHT-GEO")
print("=" * 80)
non_geo_top10_publishers = non_geo_publishers.most_common(10)
for i, (publisher, count) in enumerate(non_geo_top10_publishers, 1):
    pct_non_geo_all = 100 * count / len(non_geo_dataset_indices) if len(non_geo_dataset_indices) > 0 else 0
    pct_non_geo_with_publisher = 100 * count / non_geo_with_publisher if non_geo_with_publisher > 0 else 0
    print(
        f"{i:2d}. {publisher:40s} | {count:6d} "
        f"({pct_non_geo_all:5.1f}% von NICHT-GEO | {pct_non_geo_with_publisher:5.1f}% von NICHT-GEO mit Publisher)"
    )

print()
print("=" * 80)
print("SUMMARY")
print("=" * 80)
geo_coverage_top4_pub = sum(count for _, count in geo_top10_publishers[:4])
geo_coverage_top10_pub = sum(count for _, count in geo_top10_publishers)
non_geo_coverage_top4_pub = sum(count for _, count in non_geo_top10_publishers[:4])
non_geo_coverage_top10_pub = sum(count for _, count in non_geo_top10_publishers)

print(
    f"GEO: Top 4 Publisher decken {geo_coverage_top4_pub} Datensaetze ab "
    f"({(100 * geo_coverage_top4_pub / len(geo_dataset_indices)) if len(geo_dataset_indices) else 0:.1f}% von GEO)."
)
print(
    f"GEO: Top 10 Publisher decken {geo_coverage_top10_pub} Datensaetze ab "
    f"({(100 * geo_coverage_top10_pub / len(geo_dataset_indices)) if len(geo_dataset_indices) else 0:.1f}% von GEO)."
)
print(
    f"NICHT-GEO: Top 4 Publisher decken {non_geo_coverage_top4_pub} Datensaetze ab "
    f"({(100 * non_geo_coverage_top4_pub / len(non_geo_dataset_indices)) if len(non_geo_dataset_indices) else 0:.1f}% von NICHT-GEO)."
)
print(
    f"NICHT-GEO: Top 10 Publisher decken {non_geo_coverage_top10_pub} Datensaetze ab "
    f"({(100 * non_geo_coverage_top10_pub / len(non_geo_dataset_indices)) if len(non_geo_dataset_indices) else 0:.1f}% von NICHT-GEO)."
)

DATASET-SEGMENTIERUNG:
  Geospezifische Datensaetze: 25416
  Nicht-geospezifische Datensaetze: 126163
  Gesamtzahl Datensaetze: 151579

PUBLISHER-STATISTIK:
  GEO: Datensaetze mit Publisher: 25416
  GEO: Datensaetze ohne Publisher: 0
  GEO: Anzahl eindeutiger Publisher: 24

  NICHT-GEO: Datensaetze mit Publisher: 126163
  NICHT-GEO: Datensaetze ohne Publisher: 0
  NICHT-GEO: Anzahl eindeutiger Publisher: 57

TOP 10 PUBLISHER - GEO
 1. open-data-portal-rheinland-pfalz         |   4641 ( 18.3% von GEO |  18.3% von GEO mit Publisher)
 2. transparenzportal-hamburg                |   3484 ( 13.7% von GEO |  13.7% von GEO mit Publisher)
 3. gdi-de                                   |   3315 ( 13.0% von GEO |  13.0% von GEO mit Publisher)
 4. open-data-brandenburg                    |   3135 ( 12.3% von GEO |  12.3% von GEO mit Publisher)
 5. land-schleswig-holstein                  |   2402 (  9.5% von GEO |   9.5% von GEO mit Publisher)
 6. open-nrw                                 |   2156 (

Unser erstes Hauptstrata ist der Datentyp also geo- und nicht geospezifische Datensätze.

auf diese Unterteilt werden jetzt als zweites Hauptstrata die 4 Top-Publisher + Rest genommen.

Im Anschluss werden die Stichproben auf das Thema noch gegen geprüft um sicherzustellen, dass ein Thema nicht überproportional in der Stichprobe vertreten ist. Es wird dabei pro Datensatz nur das erste Thema berücksichtigt um einen Datensatz nicht doppelt abzubilden.

Das zweite Hauptstratum ist der Publisher durch die Annahme das zwischen den Publishern größere Qualitätsunterschiede bestehen als zwischen den Themen. Wodurch die Gesamtmenge besser abgebildet werden kann.


Die Stichprobengröße wird auf 15 pro Datensatztyp (geo- und nicht geospezifisch) festgelegt.

Insgesamt also 30 Stichproben.

Pro Datensatztyp werden dann pro Sekundärstratum (Publisher) 3 Stichproben gezogen. Bei den Top 4 Publishern also 12 Stichproben und bei den restlichen Publishern 3 Stichproben.

Zusätzlich zu der stratifizierten Zufallsstichprobe von 30 Datensätzen, werden zusätzlich 10 Datensätze gezogen die eine überdurchschnittlich schlechte Qualität aufweisen und 10 die eine überdurchschnittlich gute Qualität aufweisen um die Extremfälle in der Stichprobe zu berücksichtigen.

In [ ]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd
import requests

# ============================================================================
# Stratifizierte Zufallsstichprobe (30 Datensaetze) mit Reproduzierbarkeit
# Hauptstratum: geo / non_geo
# Sekundaerstratum: Top-4 Publisher + rest (je Hauptstratum)
# Ziehung: 3 je Sekundaerstratum => 15 je Hauptstratum => 30 gesamt
# ============================================================================

RANDOM_SEED = 42
SAMPLES_PER_SECONDARY = 3
PRIMARY_STRATA = ["geo", "non_geo"]


# --- Fallbacks, falls Funktionen aus vorherigen Zellen nicht vorhanden sind ---
if "create_session" not in globals():
    def create_session() -> requests.Session:
        session = requests.Session()
        session.headers.update(
            {
                "User-Agent": "govdata-stratified-sampler/0.1",
                "Accept": "application/rdf+xml, application/xml;q=0.9, */*;q=0.1",
            }
        )
        return session

if "sanitize_filename" not in globals():
    def sanitize_filename(value: str) -> str:
        value = (value or "").strip()
        value = re.sub(r"[^A-Za-z0-9._-]+", "-", value)
        return value.strip("-") or "dataset"

if "fetch_dataset_rdf_xml" not in globals():
    def fetch_dataset_rdf_xml(session: requests.Session, dataset_name: str, timeout: int = 60) -> bytes:
        dataset_url_patterns = [
            "https://www.govdata.de/dataset/{name}.rdf",
            "https://www.govdata.de/ckan/dataset/{name}.rdf",
            "https://govdata.de/dataset/{name}.rdf",
            "https://govdata.de/ckan/dataset/{name}.rdf",
            "https://www.govdata.de/dataset/{name}.xml",
            "https://www.govdata.de/ckan/dataset/{name}.xml",
            "https://govdata.de/dataset/{name}.xml",
            "https://govdata.de/ckan/dataset/{name}.xml",
        ]

        for pattern in dataset_url_patterns:
            url = pattern.format(name=dataset_name)
            try:
                response = session.get(url, timeout=timeout, allow_redirects=True)
            except requests.RequestException:
                continue

            if response.status_code != 200:
                continue

            content_type = (response.headers.get("Content-Type") or "").lower()
            body = response.content[:1000].lower()

            if (
                b"<rdf:rdf" in body
                or "application/rdf+xml" in content_type
                or "xml" in content_type
            ):
                return response.content

        raise RuntimeError(f"Kein RDF/XML-Endpunkt erfolgreich fuer Dataset '{dataset_name}'.")


def extract_publisher_from_row(row: pd.Series) -> str:
    """Liest Publisher aus organization.name und Fallback-Feldern."""
    org_name = row.get("organization.name")
    if isinstance(org_name, str) and org_name.strip():
        return org_name.strip()

    organization_obj = row.get("organization")
    if isinstance(organization_obj, dict):
        for key in ("name", "title", "display_name"):
            value = organization_obj.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()

    for key in ("maintainer", "author"):
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    return ""


# --- Sampling-Frame bauen ---
geo_indices = set(geo_dataset_indices)
rows = []
for idx, row in df.iterrows():
    dataset_name = row.get("name")
    if not isinstance(dataset_name, str) or not dataset_name.strip():
        continue

    publisher = extract_publisher_from_row(row)
    if not publisher:
        continue

    primary = "geo" if idx in geo_indices else "non_geo"
    rows.append(
        {
            "dataset_index": idx,
            "dataset_id": row.get("id"),
            "dataset_name": dataset_name.strip(),
            "publisher": publisher.strip(),
            "primary_stratum": primary,
        }
    )

sample_df = pd.DataFrame(rows)
if sample_df.empty:
    raise RuntimeError("Keine Datensaetze mit Name+Publisher fuer Sampling gefunden.")

print(f"Kandidaten gesamt (mit Publisher+Name): {len(sample_df)}")
print(sample_df["primary_stratum"].value_counts().to_string())
print()


# --- Top-4 Publisher je Hauptstratum bestimmen und Sekundaerstrata vergeben ---
secondary_map = {}
for primary in PRIMARY_STRATA:
    subset = sample_df[sample_df["primary_stratum"] == primary]
    top4 = subset["publisher"].value_counts().head(4).index.tolist()
    secondary_map[primary] = top4


def assign_secondary(primary: str, publisher: str) -> str:
    return publisher if publisher in secondary_map[primary] else "rest"

sample_df["secondary_stratum"] = sample_df.apply(
    lambda r: assign_secondary(r["primary_stratum"], r["publisher"]), axis=1
)


# --- Reproduzierbare Ziehung: 3 je Sekundaerstratum je Hauptstratum ---
rng = pd.Series(range(1), dtype="int64")  # nur als einfacher Seed-Anker
selected_parts = []

for primary in PRIMARY_STRATA:
    subset_primary = sample_df[sample_df["primary_stratum"] == primary].copy()
    secondaries = secondary_map[primary] + ["rest"]

    print(f"Sampling {primary.upper()}:")
    for sec in secondaries:
        stratum_df = subset_primary[subset_primary["secondary_stratum"] == sec]
        n_available = len(stratum_df)
        n_take = min(SAMPLES_PER_SECONDARY, n_available)

        if n_take == 0:
            print(f"  WARN: {sec} hat 0 Datensaetze, wird uebersprungen")
            continue

        # Deterministisch je Haupt/Sekundaer-Stratum
        local_seed = abs(hash((RANDOM_SEED, primary, sec))) % (2**31 - 1)
        picked = stratum_df.sample(n=n_take, random_state=local_seed)

        if n_take < SAMPLES_PER_SECONDARY:
            print(f"  WARN: {sec} nur {n_take} statt {SAMPLES_PER_SECONDARY} gezogen")
        else:
            print(f"  OK:   {sec} -> {n_take} gezogen")

        selected_parts.append(picked)

selected_df = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()

if selected_df.empty:
    raise RuntimeError("Sampling lieferte keine Datensaetze.")

# Falls durch leere Strata < 30: mit Rest aus gleicher Hauptgruppe auffuellen
TARGET_TOTAL = 30
if len(selected_df) < TARGET_TOTAL:
    missing = TARGET_TOTAL - len(selected_df)
    already = set(selected_df["dataset_index"])
    pool = sample_df[~sample_df["dataset_index"].isin(already)]
    if not pool.empty:
        fill = pool.sample(n=min(missing, len(pool)), random_state=RANDOM_SEED)
        fill = fill.copy()
        fill["secondary_stratum"] = fill["secondary_stratum"].astype(str) + "_fill"
        selected_df = pd.concat([selected_df, fill], ignore_index=True)

if len(selected_df) > TARGET_TOTAL:
    selected_df = selected_df.sample(n=TARGET_TOTAL, random_state=RANDOM_SEED).reset_index(drop=True)

print()
print(f"Finale Stichprobe: {len(selected_df)} Datensaetze")
print(selected_df.groupby(["primary_stratum", "secondary_stratum"]).size().to_string())
print()


# --- RDF/XML speichern ---
date_str = datetime.now().strftime("%Y-%m-%d")
out_dir = Path("data") / f"sample_{date_str}"
out_dir.mkdir(parents=True, exist_ok=True)

session = create_session()
saved = []
failed = []

# Laufnummer je (Haupt-, Sekundaerstratum)
stratum_counter = {}

for _, row in selected_df.iterrows():
    primary = row["primary_stratum"]
    secondary_raw = str(row["secondary_stratum"])
    secondary = sanitize_filename(secondary_raw.lower())
    dataset_name = row["dataset_name"]

    key = (primary, secondary)
    stratum_counter[key] = stratum_counter.get(key, 0) + 1
    seq = stratum_counter[key]

    base_name = f"{primary}_{secondary}"
    if seq == 1:
        file_name = f"{base_name}.rdf"
    else:
        file_name = f"{base_name}_{seq:02d}.rdf"

    out_path = out_dir / file_name

    try:
        rdf_xml = fetch_dataset_rdf_xml(session, dataset_name)
        out_path.write_bytes(rdf_xml)
        saved.append(
            {
                "dataset_index": row["dataset_index"],
                "dataset_id": row["dataset_id"],
                "dataset_name": dataset_name,
                "publisher": row["publisher"],
                "primary_stratum": primary,
                "secondary_stratum": secondary_raw,
                "file_path": str(out_path),
            }
        )
        print(f"OK   {dataset_name} -> {out_path}")
    except Exception as exc:
        failed.append(
            {
                "dataset_index": row["dataset_index"],
                "dataset_id": row["dataset_id"],
                "dataset_name": dataset_name,
                "error": str(exc),
            }
        )
        print(f"FAIL {dataset_name} -> {exc}")

saved_df = pd.DataFrame(saved)
failed_df = pd.DataFrame(failed)

manifest_path = out_dir / "sample_manifest.csv"
saved_df.to_csv(manifest_path, index=False)

print()
print("=" * 80)
print("ERGEBNIS")
print("=" * 80)
print(f"Gespeichert: {len(saved_df)} RDF-Dateien")
print(f"Fehler: {len(failed_df)}")
print(f"Manifest: {manifest_path}")

if not failed_df.empty:
    print("\nFehlgeschlagene Datensaetze:")
    print(failed_df[["dataset_name", "error"]].head(20).to_string(index=False))

saved_df.head(10)